## השפעת נקודה חריגה

בסעיף 11.1 זיהינו וסימנו נקודה חריגה אחת (ב-`is_outlier`), אבל מאז ההתאמות שלנו (11.6–11.9) עבדו על `df_clean` -- כלומר, **בלי** להסיר אותה. הגיע הזמן לבדוק: כמה בכלל משנה נקודה חריגה **אחת מתוך 38** להתאמה?

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()

def flag_outliers(group, col="range_measured", threshold=3.5):
    med = group[col].median()
    mad = (group[col] - med).abs().median()
    if mad == 0:
        return pd.Series(False, index=group.index)
    modified_z = 0.6745 * (group[col] - med) / mad
    return modified_z.abs() > threshold

is_outlier = df_clean.groupby("angle_deg", group_keys=False).apply(flag_outliers)
df_final = df_clean[~is_outlier].reset_index(drop=True)
print(f"df_clean: {len(df_clean)} שורות,  df_final (בלי חריגה): {len(df_final)} שורות")

In [ ]:
def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

angles = sorted(df_final["angle_deg"].unique())
x = np.array([np.sin(2*np.radians(a)) for a in angles])

y_with_outlier = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])
y_without_outlier = np.array([df_final[df_final["angle_deg"] == a]["range_measured"].mean() for a in angles])

m_with, b_with = linear_fit(x, y_with_outlier)
m_without, b_without = linear_fit(x, y_without_outlier)

print(f"עם הנקודה החריגה:  m = {m_with:.3f}, b = {b_with:.3f}")
print(f"בלי הנקודה החריגה: m = {m_without:.3f}, b = {b_without:.3f}")
print(f"שינוי בשיפוע: {abs(m_with - m_without):.3f}  ({100*abs(m_with-m_without)/m_without:.1f}%)")

נקודה חריגה **אחת** מתוך 38 (2.6% מהנתונים) הזיזה את השיפוע בכמה אחוזים -- לא עולם ומלואו, אבל בהחלט לא זניח, ובוודאי לא משהו שרוצים לגלות רק בדיעבד. שימו לב גם: השיפוע **בלי** הנקודה החריגה קרוב יותר לערך התאורטי `v0^2/g` -- רמז נוסף שהיא אכן שגיאת מדידה ולא איתות אמיתי.

### באג נפוץ: להסתמך רק על R² כדי "לפסול" נקודה חריגה

טעות שכיחה: לחשוב שאם $R^2$ נשאר גבוה (או אפילו עולה) אחרי הכללת נקודה, היא "לא בעיה". אבל $R^2$ יכול להישאר גבוה גם כשנקודה בודדת מזיזה את השיפוע משמעותית -- הוא לא תוכנן לזהות "השפעת יתר" של נקודה בודדת, רק את איכות ההתאמה הכוללת.

In [ ]:
def r_squared_of(x, y, m, b):
    resid = y - (m*x + b)
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res/ss_tot

r2_with = r_squared_of(x, y_with_outlier, m_with, b_with)
r2_without = r_squared_of(x, y_without_outlier, m_without, b_without)
print(f"R^2 עם הנקודה החריגה:  {r2_with:.4f}")
print(f"R^2 בלי הנקודה החריגה: {r2_without:.4f}")
print("שני ה-R^2 גבוהים - זה *לא* אומר שהנקודה החריגה לא השפיעה על השיפוע עצמו.")

### נסו בעצמכם

חשבו את **השארית המנורמלת** (`resid / sigma_y`, בעצם התרומה הבודדת ל-chi², לפני ריבוע) של כל נקודה בהתאמה עם הנקודה החריגה (`m_with, b_with`), והראו שהנקודה של זווית 60 בולטת משאר הנקודות.

In [ ]:
# sigma_y = ...
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
sigma_y = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])
resid_with = y_with_outlier - (m_with*x + b_with)
normalized_resid = resid_with / sigma_y
for a, r in zip(angles, normalized_resid):
    print(f"{a}°: {r:.2f}")
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "למה R^2 גבוה לא מבטיח שאין נקודה עם השפעה חריגה על השיפוע?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי R^2 תמיד שווה 1 כשיש נקודה חריגה", "correct": False, "feedback": "לא נכון."},
            {"answer": "R^2 בכלל לא מוגדר כשיש נקודות חריגות", "correct": False, "feedback": "R^2 מוגדר תמיד, גם עם נקודות חריגות."},
            {"answer": "כי R² מחושב רק על חצי מהנתונים", "correct": False, "feedback": "לא — R² מחושב על כל הנקודות בסט הנתונים, לא על מדגם חלקי."},
            {"answer": "כי R^2 מודד את איכות ההתאמה הכוללת, לא את התרומה של נקודה בודדת לשינוי הפרמטרים", "correct": True, "feedback": "נכון."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

הריצו את ההשוואה (עם/בלי הנקודה החריגה) גם על ה-`chi^2_nu` (מסעיף 11.8), עם `sigma_y` מחושב על `df_clean` ו-`df_final` בהתאמה. איזו התאמה נותנת `chi^2_nu` קרוב יותר ל-1?

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
def reduced_chi2(x, y, sigma_y, m, b):
    resid = y - (m*x + b)
    chi2 = np.sum((resid/sigma_y)**2)
    return chi2 / (len(x) - 2)

sigma_y_with = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])
sigma_y_without = np.array([
    df_final[df_final["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_final[df_final["angle_deg"] == a]))
    for a in angles
])

rchi2_with = reduced_chi2(x, y_with_outlier, sigma_y_with, m_with, b_with)
rchi2_without = reduced_chi2(x, y_without_outlier, sigma_y_without, m_without, b_without)
print(f"chi^2_nu עם הנקודה החריגה:  {rchi2_with:.3f}")
print(f"chi^2_nu בלי הנקודה החריגה: {rchi2_without:.3f}")
```
`````